# NS-3 Assignment 6 Analysis Notebook
Run `run_all_experiments.sh` first. This notebook reads trace files from `output/` and creates the requested cWnd, drop-marker and throughput plots.


In [ ]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
OUTPUT = Path('output')
FIGDIR = OUTPUT / 'figures'
FIGDIR.mkdir(parents=True, exist_ok=True)

def read_cwnd(path):
    p = Path(path)
    if not p.exists() or p.stat().st_size == 0:
        return pd.DataFrame(columns=['time','old','new'])
    return pd.read_csv(p, sep='\t', names=['time','old','new'])

def read_drops(path):
    p = Path(path)
    if not p.exists() or p.stat().st_size == 0:
        return pd.DataFrame(columns=['time'])
    return pd.read_csv(p, sep='\t', names=['time'])

def read_thr(path):
    p = Path(path)
    if not p.exists() or p.stat().st_size == 0:
        return pd.DataFrame(columns=['time','mbps'])
    return pd.read_csv(p, sep='\t', names=['time','mbps'])


## Task 6.1.4 Plot 1: fixed delay = 10 ms, compare TCP variants and BER


In [ ]:

plt.figure(figsize=(12, 6))
for tcp in ['TcpNewReno', 'TcpVegas']:
    for ber in ['0.00001', '0.000001']:
        exp = f'{tcp}_ber_{ber}_delay_10ms'
        cwnd = read_cwnd(OUTPUT / f'614_{exp}.cwnd')
        drops = read_drops(OUTPUT / f'614_{exp}.rxdrop')
        if not cwnd.empty:
            plt.plot(cwnd.time, cwnd.new / 1040, label=f'{tcp}, BER={ber}')
            for t in drops.time:
                plt.axvline(t, linestyle='--', linewidth=0.7, alpha=0.35)
plt.xlabel('Time [s]')
plt.ylabel('Congestion window [packets]')
plt.title('Task 6.1.4: fixed delay 10 ms, cWnd over time; dashed lines mark packet drops')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGDIR / '614_fixed_delay_10ms.png', dpi=200)
plt.show()


## Task 6.1.4 Plot 2: fixed BER = 1e-5, compare TCP variants and delay


In [ ]:

plt.figure(figsize=(12, 6))
for tcp in ['TcpNewReno', 'TcpVegas']:
    for delay in ['10ms', '20ms']:
        exp = f'{tcp}_ber_0.00001_delay_{delay}'
        cwnd = read_cwnd(OUTPUT / f'614_{exp}.cwnd')
        drops = read_drops(OUTPUT / f'614_{exp}.rxdrop')
        if not cwnd.empty:
            plt.plot(cwnd.time, cwnd.new / 1040, label=f'{tcp}, delay={delay}')
            for t in drops.time:
                plt.axvline(t, linestyle='--', linewidth=0.7, alpha=0.35)
plt.xlabel('Time [s]')
plt.ylabel('Congestion window [packets]')
plt.title('Task 6.1.4: fixed BER 1e-5, cWnd over time; dashed lines mark packet drops')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGDIR / '614_fixed_ber_1e-5.png', dpi=200)
plt.show()


## Task 6.1.5 Analysis template
NewReno normally shows slow start followed by additive increase and multiplicative decrease. When packet drops occur, the cWnd decreases sharply and then grows again in congestion avoidance. A higher BER usually causes more frequent reductions, so the average cWnd becomes smaller. A larger delay increases RTT, so cWnd growth reacts more slowly and the flow takes longer to recover. Vegas is delay-based and tends to keep a smoother and often smaller cWnd because it reacts to increasing queueing delay before packet loss occurs.


## Task 6.2.3 / 6.2.5 / 6.2.6: multi-flow cWnd plots


In [ ]:

def plot_multi(exp, title):
    plt.figure(figsize=(12, 6))
    for flow in [1, 2]:
        cwnd = read_cwnd(OUTPUT / f'623_{exp}_flow{flow}.cwnd')
        if not cwnd.empty:
            plt.plot(cwnd.time, cwnd.new / 1040, label=f'flow {flow}')
    plt.xlabel('Time [s]')
    plt.ylabel('Congestion window [packets]')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGDIR / f'{exp}_cwnd.png', dpi=200)
    plt.show()

for tcp in ['TcpBic', 'TcpNewReno']:
    plot_multi(f'{tcp}_equal', f'Task 6.2.3: {tcp}, equal conditions')
    plot_multi(f'{tcp}_delaychange', f'Task 6.2.5: {tcp}, Receiver 1 delay changed at 20 s')
    plot_multi(f'{tcp}_udp', f'Bonus Task 6.2.6: {tcp}, UDP flow starts at 30 s')


## Bonus Task 6.2.4: throughput over time


In [ ]:

def plot_throughput(exp, title):
    plt.figure(figsize=(12, 6))
    for flow in [1, 2]:
        thr = read_thr(OUTPUT / f'624_{exp}_flow{flow}.throughput')
        if not thr.empty:
            plt.plot(thr.time, thr.mbps, marker='o', markersize=2, label=f'flow {flow}')
    plt.xlabel('Time [s]')
    plt.ylabel('Throughput [Mbit/s]')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGDIR / f'{exp}_throughput.png', dpi=200)
    plt.show()

for tcp in ['TcpBic', 'TcpNewReno']:
    for suffix in ['equal', 'delaychange', 'udp']:
        exp = f'{tcp}_{suffix}'
        plot_throughput(exp, f'Throughput: {exp}')


## Multi-flow analysis template
When flow 2 starts at 11 s, both flows must share the 10 Mbit/s bottleneck. Flow 1 therefore loses part of its previously available capacity and its cWnd typically decreases or oscillates more strongly. Under equal conditions, the two flows should converge toward a roughly fair share, though the exact transient differs by TCP variant. BIC is more aggressive and can show faster growth and stronger oscillation, while NewReno usually follows a more regular AIMD pattern. When Receiver 1's access-link delay is increased at 20 s, flow 1 obtains a larger RTT and can become disadvantaged because its window growth per unit time slows. When the UDP flow starts at 30 s, it consumes bottleneck capacity without TCP-style backoff, so TCP cWnd values usually drop and throughput is reduced.
